## Condición de Neumann

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurimendiluce/AN2026/blob/main/diferencias_finitas/clase_4.ipynb)

### Problema

Consideramos la ecuación del calor en $[0,1]$ con una condición Dirichlet en $x=1$ y una condición **Neumann** en $x=0$:

$$
\begin{aligned}
u_t &= u_{xx}, & 0<x<1,\ t>0,\\
u(1,t) &= 0,\\
u_x(0,t) &= g(t),\\
u(x,0) &= u_0(x).
\end{aligned}
$$

La pregunta que queremos ilustrar computacionalmente es: **¿cómo se discretiza la condición de Neumann?** Vamos a comparar tres alternativas:

1. **Forward (hacia el interior)**: usar solo el punto vecino $u_1$ para aproximar $u_x(0,t)$
1. **Backward (nodo ficticio)**: introducir el nodo ficticio $u_{-1}$ para aproximar $u_x(0,t)$.
2. **Centradas (nodo ficticio)**: introducir un punto ficticio $u_{-1}$ para aproximar $u_x(0,t)$.

In [1]:
using LinearAlgebra
using Printf
using Plots

Para validar numéricamente ambos esquemas usamos la solución exacta

$$
u(x,t) = e^{-t}\sin(1-x),
$$

que satisface $u_t=u_{xx}$ para cualquier autovalor (chequeo directo: $u_t=-e^{-t}\sin(1-x)=u_{xx}$), y además

$$
u(1,t) = e^{-t}\sin(0) = 0, \qquad u_x(0,t) = -e^{-t}\cos(1) =: g(t).
$$

Así, $g(t) = -\cos(1)\,e^{-t}$ y $u_0(x) = \sin(1-x)$.

In [3]:
u_exacta(x, t) = exp(-t) * sin(1 - x)

g(t) = -cos(1.0) * exp(-t)

u0(x) = sin(1 - x)

u0 (generic function with 1 method)

### Armando el esquema

Usamos el método explícito estándar (Euler hacia adelante en tiempo, diferencias
centradas en espacio) con $r = \Delta t/\Delta x^2$:

$$
U_j^{n+1} = U_j^n + r\left(U_{j-1}^n - 2U_j^n + U_{j+1}^n\right).
$$

La condición Dirichlet $u(1,t)=0$ simplemente fija $U_N \equiv 0$ y no agrega incógnitas.
Lo que cambia entre los tres esquemas es **cómo tratamos el nodo $x_0=0$**, donde
la condición de Neumann $u_x(0,t)=g(t)$ debe combinarse con el esquema interior.

---

**1. Forward, sin nodo ficticio.**
Aproximamos $u_x(0,t)\approx (u_1-u_0)/\Delta x = g(t)$ y despejamos $u_0$:

$$
u_0^n = u_1^n - \Delta x\, g(t^n).
$$

Con esto $u_0$ deja de ser una incógnita: el vector incógnita arranca en $U_1$. Sustituyendo
en la fórmula del esquema para $j=1$:

$$
U_1^{n+1} = (1-r)U_1^n + r\,U_2^n - r\Delta x\, g(t^n).
$$

Esta condición de contorno es de **primer orden** ($O(\Delta x)$ de error de truncamiento local).

---

**2. Backward, con nodo ficticio.**
Introducimos el nodo ficticio $x_{-1}=-\Delta x$ y aplicamos la fórmula interior también en $j=0$:

$$
U_0^{n+1} = U_0^n + r\left(U_{-1}^n - 2U_0^n + U_1^n\right).
$$

Para cerrar el sistema aproximamos $u_x(0,t)$ con una diferencia **backward** de primer orden
usando el nodo ficticio:

$$
\frac{u_0^n - u_{-1}^n}{\Delta x} = g(t^n)
\quad\Longrightarrow\quad
u_{-1}^n = u_0^n - \Delta x\, g(t^n).
$$

Sustituyendo:

$$
U_0^{n+1} = (1-r)U_0^n + r\,U_1^n - r\Delta x\, g(t^n).
$$

Notar que la fila queda idéntica a la del esquema forward, pero ahora $U_0$ **sí** es una
incógnita del sistema (el vector arranca en $U_0$, no en $U_1$). Sigue siendo de primer orden.

---

**3. Centradas, con nodo ficticio.**
Mismo nodo ficticio $x_{-1}$, pero ahora aproximamos $u_x(0,t)$ con una diferencia
**centrada** (segundo orden):

$$
\frac{u_1^n - u_{-1}^n}{2\Delta x} = g(t^n)
\quad\Longrightarrow\quad
u_{-1}^n = u_1^n - 2\Delta x\, g(t^n).
$$

Sustituyendo en la fórmula interior evaluada en $j=0$:

$$
U_0^{n+1} = U_0^n + r\left(u_1^n - 2\Delta x g(t^n) - 2U_0^n + U_1^n\right)
= (1-2r)U_0^n + 2r\,U_1^n - 2r\Delta x\, g(t^n).
$$

Esta discretización de la condición de contorno es de **segundo orden**
($O(\Delta x^2)$), y es la que esperamos que dé mejor convergencia global.

En los tres casos usamos $r\le 1/2$ para que el esquema explícito sea estable
(la modificación de la primera fila no cambia esta cota, ya que se puede verificar
por dominancia diagonal / principio del máximo discreto).

In [ ]:
function neumann_calor(u0, g, r, N, esquema::Symbol; steps=50)
    Δx = 1.0 / N
    Δt = r * Δx^2
    x = 0:Δx:1   # x_0,...,x_N

    if esquema == :forward
        # Incógnitas: U_1,...,U_{N-1}. u_0 se elimina vía (u_1-u_0)/Δx = g(t)
        Nu = N - 1
        U  = u0.(x[2:end-1])
        A  = Tridiagonal(ones(Nu-1), -2*ones(Nu), ones(Nu-1))
        M  = Matrix(I, Nu, Nu) + r*A
        M[1,1] = 1 - r
        for n in 0:steps-1
            tn = n*Δt
            F = zeros(Nu)
            F[1] = -r*Δx*g(tn)
            U = M*U + F
        end
        x_u = x[2:end-1]

    elseif esquema == :backward
        # Incógnitas: U_0,...,U_{N-1}. Ghost: u_{-1} = u_0 - Δx*g(t)
        Nu = N
        U  = u0.(x[1:end-1])
        A  = Tridiagonal(ones(Nu-1), -2*ones(Nu), ones(Nu-1))
        M  = Matrix(I, Nu, Nu) + r*A
        M[1,1] = 1 - r
        for n in 0:steps-1
            tn = n*Δt
            F = zeros(Nu)
            F[1] = -r*Δx*g(tn)
            U = M*U + F
        end
        x_u = x[1:end-1]

    elseif esquema == :centradas
        # Incógnitas: U_0,...,U_{N-1}. Ghost: u_{-1} = u_1 - 2Δx*g(t)
        Nu = N
        U  = u0.(x[1:end-1])
        A  = Tridiagonal(ones(Nu-1), -2*ones(Nu), ones(Nu-1))
        M  = Matrix(I, Nu, Nu) + r*A
        M[1,1] = 1 - 2r
        M[1,2] = 2r
        for n in 0:steps-1
            tn = n*Δt
            F = zeros(Nu)
            F[1] = -2r*Δx*g(tn)
            U = M*U + F
        end
        x_u = x[1:end-1]

    else
        error("esquema desconocido: $esquema")
    end

    return x_u, U
end

neumann_calor (generic function with 1 method)

### Comparación visual de las soluciones

Corremos los tres esquemas con los mismos $N$, $r$ y número de pasos, y los
graficamos junto con la solución exacta en $t_{\text{final}}$. Para el esquema
`:forward`, como $u_0$ no es incógnita, lo reconstruimos a partir de la
misma fórmula que usamos para eliminarlo.

In [ ]:
N = 40
r = 0.4
steps = 200

xf, Uf = neumann_calor(u0, g, r, N, :forward;   steps=steps)
xb, Ub = neumann_calor(u0, g, r, N, :backward;  steps=steps)
xc, Uc = neumann_calor(u0, g, r, N, :centradas; steps=steps)

Δx = 1.0/N
Δt = r*Δx^2
t_final = steps*Δt

# reconstruimos u_0 para :forward (fue eliminado como incógnita)
u0_forward = Uf[1] - Δx*g(t_final)

xf_full = vcat(0.0, collect(xf), 1.0);  Uf_full = vcat(u0_forward, Uf, 0.0)
xb_full = vcat(collect(xb), 1.0);       Ub_full = vcat(Ub, 0.0)
xc_full = vcat(collect(xc), 1.0);       Uc_full = vcat(Uc, 0.0)

xx = range(0, 1, length=400)
ue = u_exacta.(xx, t_final)

# Subplot 1: forward
p1 = plot(xx, ue, label="exacta", lw=2, color=:black,ls = :dash)
plot!(p1, xf_full, Uf_full, label="forward (orden 1)", color=:blue)
xlabel!(p1, "x"); ylabel!(p1, "u(x, t_final)")
title!(p1, "Forward")

# Subplot 2: backward ficticio
p2 = plot(xx, ue, label="exacta", lw=2, color=:black,ls = :dash)
plot!(p2, xb_full, Ub_full, label="backward ficticio (orden 1)", color=:red)
xlabel!(p2, "x"); ylabel!(p2, "u(x, t_final)")
title!(p2, "Backward ficticio")

# Subplot 3: centradas ficticio
p3 = plot(xx, ue, label="exacta", lw=2, color=:black,ls = :dash)
plot!(p3, xc_full, Uc_full, label="centradas ficticio (orden 2)", color=:green)
xlabel!(p3, "x"); ylabel!(p3, "u(x, t_final)")
title!(p3, "Centradas ficticio")

# Subplot 4: los tres esquemas juntos (o dejalo vacío / comparación general)
#p4 = plot(xx, ue, label="exacta", lw=2, color=:black)
#plot!(p4, xf_full, Uf_full, seriestype=:scatter, label="forward", marker=:circle, color=:blue)
#plot!(p4, xb_full, Ub_full, seriestype=:scatter, label="backward", marker=:diamond, color=:red)
#plot!(p4, xc_full, Uc_full, seriestype=:scatter, label="centradas", marker=:star5, color=:green)
#xlabel!(p4, "x"); ylabel!(p4, "u(x, t_final)")
#title!(p4, "Comparación")

# Combinar en 2x2
plot(p1, p2, p3, layout=(2,2), size=(900,700),
     plot_title="Esquemas de Neumann, t = $(round(t_final, digits=4)), N = $N, r = $r",
     legend=:best, legendfontsize=6)

### Estudio de convergencia

Para cada esquema, fijamos $r$ y el tiempo final $T$, refinamos $N$ (duplicando
cada vez) y medimos el error en norma $\infty$ contra la solución exacta.
Estimamos el orden con el cociente logarítmico local

$$
p_i \approx \frac{\log(E_{i-1}/E_i)}{\log(N_i/N_{i-1})},
$$

que compara cada par consecutivo de refinamientos (preferible al ajuste global,
porque muestra si el error de redondeo empieza a dominar en $N$ grandes).

In [ ]:
function error_convergencia(esquema; r=0.4, T=0.1, Ns=[10,20,40,80,160])
    errores = Float64[]
    for N in Ns
        Δx = 1.0/N
        Δt = r*Δx^2
        steps = round(Int, T/Δt)
        x_u, U = neumann_calor(u0, g, r, N, esquema; steps=steps)
        t_final = steps*Δt
        uex = u_exacta.(collect(x_u), t_final)
        push!(errores, maximum(abs.(U .- uex)))
    end
    return errores
end

Ns = [10, 20, 40, 80, 160]
r  = 0.4
T  = 0.1

errores_esquemas = Dict{Symbol, Vector{Float64}}()

for esquema in (:forward, :backward, :centradas)
    errores = error_convergencia(esquema; r=r, T=T, Ns=Ns)
    errores_esquemas[esquema] = errores
    println("\n=== esquema: $esquema ===")
    @printf("%6s  %12s  %10s\n", "N", "err_inf", "orden α")
    for i in 1:length(Ns)
        if i == 1
            @printf("%6d  %12.3e  %10s\n", Ns[i], errores[i], "--")
        else
            α = log(errores[i-1]/errores[i]) / log(Ns[i]/Ns[i-1])
            @printf("%6d  %12.3e  %10.3f\n", Ns[i], errores[i], α)
        end
    end
end

### Conclusión

Los resultados numéricos confirman lo que predice el análisis: el orden de
convergencia global coincide con el orden de truncamiento local de la
discretización de la condición de Neumann en $x=0$.

- **`:forward`** y **`:backward`** (ambas discretizaciones de primer orden de
  $u_x(0,t)=g(t)$) convergen con $p\approx 1$.
- **`:centradas`** (discretización de segundo orden vía nodo ficticio)
  converge con $p\approx 2$, y además el error es varios órdenes de magnitud
  menor a igual $N$.


### ¿Por qué forward/backward "convergen" a orden 1 si la fila del borde no es consistente?

Si tomamos la ecuación de actualización de $U_0$ para los esquemas `:forward`/`:backward`,

$$
U_0^{n+1} = (1-r)U_0^n + r\,U_1^n - r\Delta x\, g(t^n),
$$

y calculamos el error de truncamiento local de la forma habitual,
$\tau_0^n = \dfrac{u(0,t^{n+1}) - \big[(1-r)u(0,t^n)+r\,u(\Delta x,t^n)-r\Delta x\,g(t^n)\big]}{\Delta t}$,
Taylor da (a $r$ fijo, $\Delta t=r\Delta x^2$):

$$
\tau_0^n \;\longrightarrow\; \tfrac12\,u_{xx}(0,t^n) \quad \text{cuando } \Delta x\to 0,
$$

es decir, **no tiende a cero**: el esquema no es consistente puntualmente en $x=0$. Esto contrasta
con `:centradas`, donde la aproximación del nodo ficticio es un orden más precisa
($u_{-1}$ tiene error $O(\Delta x^3)$ en vez de $O(\Delta x^2)$) y en ese caso sí se cumple
$\tau_0^n = O(\Delta x)\to 0$.

Esto genera una aparente paradoja: si $\tau_0=O(1)$ en una fila, el argumento ingenuo
"consistencia + estabilidad ⟹ convergencia" (acotando con $\|\tau\|_\infty$) predeciría
error global $O(1)$, no una convergencia de orden 1 como la que medimos antes. Lo que
realmente ocurre es más fino: ese error $O(1)$ está confinado a **un solo nodo** de los
$N\sim 1/\Delta x$, y se agrega con un $\Delta t\cdot\tau_0=O(\Delta t)$ en cada paso.
La estabilidad del esquema amortigua y dispersa esa perturbación
puntual en vez de dejarla acumularse sin control, y el resultado neto es que su aporte
al error global es $O(\Delta x)$: el esquema pasa de "no consistente en esa fila" (orden 0)
a "orden 1 global", no a orden 2 como el resto de la malla.

Verificamos esto en dos partes: (1) numéricamente, que $\tau_0$ no converge a cero para
forward/backward pero sí para centradas; (2) visualmente, que el error numérico
$U-u_{\text{exacta}}$ queda concentrado cerca de $x=0$ para forward/backward, y es chico
y parejo para centradas.

In [ ]:
function tau_borde(esquema::Symbol; r=0.4, t0=0.3, Ns=[10,20,40,80,160,320,640])
    taus = Float64[]
    for N in Ns
        Δx = 1.0/N
        Δt = r*Δx^2
        u0n   = u_exacta(0.0, t0)
        u1n   = u_exacta(Δx, t0)
        u0np1 = u_exacta(0.0, t0+Δt)
        if esquema == :forward_backward
            residuo = u0np1 - ((1-r)*u0n + r*u1n - r*Δx*g(t0))
        elseif esquema == :centradas
            residuo = u0np1 - ((1-2r)*u0n + 2r*u1n - 2r*Δx*g(t0))
        else
            error("esquema desconocido")
        end
        push!(taus, residuo/Δt)
    end
    return taus
end

u_xx_exacta(x,t) = -exp(-t)*sin(1-x)   # = u_t = u_xx, por la EDP

Ns = [10, 20, 40, 80, 160, 320, 640]
r  = 0.4
t0 = 0.3

tau_fb = tau_borde(:forward_backward; r=r, t0=t0, Ns=Ns)
tau_c  = tau_borde(:centradas;        r=r, t0=t0, Ns=Ns)

println("     N   τ₀ forward/backward     τ₀ centradas")
for i in eachindex(Ns)
    @printf("%6d      %14.6f      %14.6f\n", Ns[i], tau_fb[i], tau_c[i])
end

limite_teorico = 0.5*u_xx_exacta(0.0, t0)
println("\nLímite teórico τ₀(forward/backward) → (1/2)u_xx(0,t0) = ", round(limite_teorico, digits=6))


### Efecto visible en el borde: perfil del error en $x$

Ahora graficamos directamente $U-u_{\text{exacta}}$ en función de $x$ (no solo su norma
$\infty$), con $N$ fijo, para las tres discretizaciones. Si la explicación anterior es
correcta, deberíamos ver el error de forward/backward más grande y concentrado cerca de
$x=0$, mientras que en centradas debería ser chico y repartido de forma más pareja en
todo el dominio.

In [ ]:
N = 80
r = 0.4
steps = 400

xf, Uf = neumann_calor(u0, g, r, N, :forward;   steps=steps)
xb, Ub = neumann_calor(u0, g, r, N, :backward;  steps=steps)
xc, Uc = neumann_calor(u0, g, r, N, :centradas; steps=steps)

Δx = 1.0/N
Δt = r*Δx^2
t_final = steps*Δt

u0_forward = Uf[1] - Δx*g(t_final)  # reconstruimos u_0, eliminado como incógnita en :forward

xf_full = vcat(0.0, collect(xf), 1.0);  Uf_full = vcat(u0_forward, Uf, 0.0)
xb_full = vcat(collect(xb), 1.0);       Ub_full = vcat(Ub, 0.0)
xc_full = vcat(collect(xc), 1.0);       Uc_full = vcat(Uc, 0.0)

err_f = Uf_full .- u_exacta.(xf_full, t_final)
err_b = Ub_full .- u_exacta.(xb_full, t_final)
err_c = Uc_full .- u_exacta.(xc_full, t_final)

plot(xf_full, err_f,  label="forward",              lw=1.5)
plot!(xb_full, err_b, label="backward ficticio",   lw=1.5)
plot!(xc_full, err_c,   label="centradas ficticio",  lw=1.5)
xlabel!("x"); ylabel!("error  U - u_exacta")
title!("Perfil del error en x,  N=$N,  t=$(round(t_final, digits=4))")

**Lectura de los resultados esperados:** la tabla de $\tau_0$ debería mostrar que
`forward`/`backward` se estabilizan en un valor no nulo (cercano al límite teórico) apenas
$N$ crece, mientras que `centradas` decae hacia cero. En el gráfico de perfil, el error de
`forward`/`backward` debería ser visiblemente mayor cerca de $x=0$ y decrecer hacia
$x=1$ (donde la condición Dirichlet fija el error exactamente en cero), mientras que
`centradas` debería quedar chico en todo el dominio. Esto es justamente la firma de una
inconsistencia puntual confinada al borde, amortiguada —pero no eliminada— por la
estabilidad del esquema.